In [1]:
import numpy as np
import pandas as pd
import plotly.offline as pyo
import plotly.graph_objs as go

In [2]:
match = pd.read_csv('matches.csv')
delivery = pd.read_csv('deliveries.csv')

ipl = delivery.merge(match, left_on='match_id', right_on='id')
ipl.head(3)

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN


In [3]:
# scatter plots are drawn between to continous variable
# Problem :- We are going to draw a scatter plot between Batsman Avg(X axis and )
# Batsman Stike Rate (Y axis) of the top 50 batsman in IPL ( all time)

In [4]:
top50 = ipl.groupby('batsman')['batsman_runs'].sum().sort_values(ascending=False).head(50)
new_ipl = ipl[ipl['batsman'].isin(top50.index)]
top50.head()

batsman
SK Raina     4548
V Kohli      4423
RG Sharma    4207
G Gambhir    4132
DA Warner    4014
Name: batsman_runs, dtype: int64

In [5]:
runs = new_ipl.groupby('batsman')['batsman_runs'].sum()
balls = new_ipl.groupby('batsman')['batsman_runs'].count()

sr = (runs/balls) * 100

sr = sr.reset_index()
sr.head()

,batsman,batsman_runs
0,AB de Villiers,145.129059
1,AC Gilchrist,133.054662
2,AJ Finch,126.299213
3,AM Rahane,117.486549
4,AT Rayudu,123.014257


In [6]:
# Calculate the Avg
# Avg = (Total Number of Runs) / (Number of outs)

# calculating number of outs for top 50 batsman

out = ipl[ipl['player_dismissed'].isin(top50.index)]
n_outs = out['player_dismissed'].value_counts()

avg = runs/n_outs
avg = avg.reset_index()
avg  =avg.rename(columns={'index':'batsman',0:'avg'})
avg = avg.merge(sr,on='batsman')
avg.head()

,batsman,avg,batsman_runs
0,AB de Villiers,38.307692,145.129059
1,AC Gilchrist,27.223684,133.054662
2,AJ Finch,27.186441,126.299213
3,AM Rahane,33.593407,117.486549
4,AT Rayudu,27.146067,123.014257


In [7]:
# scatter plot
trace1 = go.Scatter( x = avg['avg'], y=avg['batsman_runs'],
                    mode='markers', text=avg['batsman'],
                    marker={'color':'#00a65a','size':16})

data = [trace1]
layout = go.Layout(title='Batsman Avg VS SR',
                    xaxis={'title' : 'Batsman Average'},
                    yaxis={'title' : 'Batsman Strike Rate'})
fig = go.Figure(data,layout)

pyo.plot(fig)

'temp-plot.html'

#### Line Plot

In [8]:
# Year by Year batsman performance
single = ipl[ipl['batsman'] == 'V Kohli']
performance = single.groupby('season')['batsman_runs'].sum().reset_index()
performance

single1 = ipl[ipl['batsman'] == 'MS Dhoni']
performance1 = single1.groupby('season')['batsman_runs'].sum().reset_index()
performance1

,season,batsman_runs
0,2008,414
1,2009,332
2,2010,287
3,2011,392
4,2012,357
5,2013,461
6,2014,371
7,2015,372
8,2016,284
9,2017,290


In [9]:
trace1 = go.Scatter(x = performance['season'] , y=performance['batsman_runs'],
                    mode='lines+markers', marker={'color':'red'}, name='Virat Kohli')

trace2 = go.Scatter(x = performance1['season'] , y=performance1['batsman_runs'],
                    mode='lines+markers', marker={'color':'blue'}, name='MS Dhoni')

data = [trace1,trace2]
layout = go.Layout(title='Line plot of V Kohli"s IPL Season VS Total Runs Scored',
                    xaxis = {'title':'IPL Season Year'},
                    yaxis={'title':'Runs scored in a year'})

fig = go.Figure(data,layout)
pyo.plot(fig)


'temp-plot.html'

In [10]:
# Multiple line charts
def batsman_comp(*name):
    data = []
    for i in name:
        single = ipl[ipl['batsman']==i]
        performance = single.groupby('season')['batsman_runs'].sum().reset_index()
        trace = go.Scatter(x = performance['season'], y=performance['batsman_runs'],
                            mode = 'lines+markers', name=i)
        data.append(trace)
    layout = go.Layout(title='Line plot of V Kohli"s IPL Season VS Total Runs Scored',
                    xaxis = {'title':'IPL Season Year'},
                    yaxis={'title':'Runs scored in a year'})
    fig = go.Figure(data,layout)
    pyo.plot(fig)

In [11]:
ipl['batsman'].value_counts().head(10)

batsman
V Kohli       3494
G Gambhir     3433
SK Raina      3369
RG Sharma     3274
S Dhawan      3005
RV Uthappa    2960
DA Warner     2902
MS Dhoni      2680
AM Rahane     2602
CH Gayle      2532
Name: count, dtype: int64

In [12]:
batsman_comp('V Kohli','G Gambhir','SK Raina', 'RG Sharma', 'S Dhawan')

#### Bar Plot

In [13]:
top10 = ipl.groupby('batsman')['batsman_runs'].sum().sort_values(ascending=False).head(10)
top10_df = ipl[ipl['batsman'].isin(top10.index)]
top10_df.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
